In [1]:
import pandas as pd
import statsmodels.api as sm

In [2]:
def backward_elimination(y, X, significance_level=0.10):
    """
    Iteratively removes variables with p-values > significance_level.
    
    """
    X = sm.add_constant(X)  # add intercept
    model = sm.OLS(y, X).fit()
    
    while True:
        # Get max p-value
        p_values = model.pvalues
        max_pval = p_values.max()
        worst_var = p_values.idxmax()
        
        # Stop if all p-values are <= significance_level
        if max_pval <= significance_level:
            break
        
        # Do not drop the constant
        if worst_var == "const":
            break
        
        # Drop worst variable
        print(f"Dropping '{worst_var}' (p-value = {max_pval:.4f})")
        X = X.drop(columns=[worst_var])
        
        # Refit model
        model = sm.OLS(y, X).fit()
    
    return model, X

# Benin

In [3]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataHousingCPI/Benin_completes"
df = pd.read_excel(x+".xlsx")

In [4]:
# Define X and y
X = df.drop(columns=["House CPI", "Months"])   
X = sm.add_constant(X)       
y = df["House CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.6949)
Dropping 'EUR/USD' (p-value = 0.1547)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.1269)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.2925)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                      -0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                       nan
Date:                Fri, 10 Oct 2025   Prob (F-statistic):                nan
Time:                        15:35:18   Log-Likelihood:                -697.15
No. Observations:                 302   AIC:                             1396.
Df Residuals:                     301   BIC:                             1400.
Df Model:                           0                                         
Covariance Type:            nonrobust                           

In [5]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.715
Model:                            OLS   Adj. R-squared:                  0.710
Method:                 Least Squares   F-statistic:                     148.1
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           3.00e-78
Time:                        15:35:18   Log-Likelihood:                -506.04
No. Observations:                 301   AIC:                             1024.
Df Residuals:                     295   BIC:                             1046.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
lag_y               

In [6]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'EUR/USD' (p-value = 0.8569)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.6494)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.5203)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.9576)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.715
Model:                            OLS   Adj. R-squared:                  0.714
Method:                 Least Squares   F-statistic:                     748.3
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           2.23e-83
Time:                        15:35:18   Log-Likelihood:                -506.38
No. Observations:                 301   AIC:                             1017.
Df Residuals:                     299   BIC:                             1024.
Df Model:                           1                                         
Covariance Type:            nonrobust                           

In [7]:
# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())

                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.018
Model:                            OLS   Adj. R-squared:                  0.004
Method:                 Least Squares   F-statistic:                     1.319
Date:                Fri, 10 Oct 2025   Prob (F-statistic):              0.263
Time:                        15:35:18   Log-Likelihood:                -692.37
No. Observations:                 301   AIC:                             1395.
Df Residuals:                     296   BIC:                             1413.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_const   

In [8]:
ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.5773)
Dropping 'lag_EUR/USD' (p-value = 0.4662)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     2.226
Date:                Fri, 10 Oct 2025   Prob (F-statistic):              0.110
Time:                        15:35:18   Log-Likelihood:                -692.80
No. Observations:                 301   AIC:                             1392.
Df Residuals:                     298   BIC:                             1403.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025   

In [9]:
# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.717
Model:                            OLS   Adj. R-squared:                  0.712
Method:                 Least Squares   F-statistic:                     149.6
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           1.02e-78
Time:                        15:35:18   Log-Likelihood:                -504.94
No. Observations:                 301   AIC:                             1022.
Df Residuals:                     295   BIC:                             1044.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_y       

In [10]:
ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.6907)
Dropping 'lag_EUR/USD' (p-value = 0.3053)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.2137)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.7896)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.715
Model:                            OLS   Adj. R-squared:                  0.714
Method:                 Least Squares   F-statistic:                     748.3
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           2.23e-83
Time:                        15:35:18   Log-Likelihood:                -506.38
No. Observations:                 301   AIC:                             1017.
Df Residuals:                     299   BIC:                             1024.
Df Model:                           1                                         
Covariance Type:            nonrobust           

# Burkina

In [11]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataHousingCPI/Burkina_completes"
df = pd.read_excel(x+".xlsx")


In [12]:

# Define X and y
X = df.drop(columns=["House CPI", "Months"])   
X = sm.add_constant(X)       
y = df["House CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())




Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.075
Model:                            OLS   Adj. R-squared:                  0.062
Method:                 Least Squares   F-statistic:                     6.015
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           0.000115
Time:                        15:35:19   Log-Likelihood:                -553.28
No. Observations:                 302   AIC:                             1117.
Df Residuals:                     297   BIC:                             1135.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------------------

In [13]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.903
Model:                            OLS   Adj. R-squared:                  0.901
Method:                 Least Squares   F-statistic:                     546.9
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          7.39e-147
Time:                        15:35:19   Log-Likelihood:                -212.84
No. Observations:                 301   AIC:                             437.7
Df Residuals:                     295   BIC:                             459.9
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
lag_y               

In [14]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.8771)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.7652)
Dropping 'EUR/USD' (p-value = 0.2497)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.902
Model:                            OLS   Adj. R-squared:                  0.901
Method:                 Least Squares   F-statistic:                     1374.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          3.95e-151
Time:                        15:35:19   Log-Likelihood:                -213.57
No. Observations:                 301   AIC:                             433.1
Df Residuals:                     298   BIC:                             444.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                             coef    s

In [15]:

# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.080
Model:                            OLS   Adj. R-squared:                  0.067
Method:                 Least Squares   F-statistic:                     6.425
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           5.71e-05
Time:                        15:35:19   Log-Likelihood:                -550.84
No. Observations:                 301   AIC:                             1112.
Df Residuals:                     296   BIC:                             1130.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_const   

In [16]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.1098)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.072
Model:                            OLS   Adj. R-squared:                  0.063
Method:                 Least Squares   F-statistic:                     7.669
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           5.94e-05
Time:                        15:35:19   Log-Likelihood:                -552.14
No. Observations:                 301   AIC:                             1112.
Df Residuals:                     297   BIC:                             1127.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------

In [17]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.902
Model:                            OLS   Adj. R-squared:                  0.900
Method:                 Least Squares   F-statistic:                     541.2
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          2.97e-146
Time:                        15:35:19   Log-Likelihood:                -214.26
No. Observations:                 301   AIC:                             440.5
Df Residuals:                     295   BIC:                             462.8
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_y       

In [18]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.6007)
Dropping 'lag_EUR/USD' (p-value = 0.5677)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.5807)
Dropping 'lag_const' (p-value = 0.2050)

Final model summary:
                                 OLS Regression Results                                
Dep. Variable:              House CPI   R-squared (uncentered):                   0.907
Model:                            OLS   Adj. R-squared (uncentered):              0.907
Method:                 Least Squares   F-statistic:                              1463.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):                   3.71e-155
Time:                        15:35:19   Log-Likelihood:                         -215.53
No. Observations:                 301   AIC:                                      435.1
Df Residuals:                     299   BIC:                                      442.5
Df Model:                           2                                       

# Côte d'Ivoire

In [19]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataHousingCPI/Ivoire"
df = pd.read_excel(x+".xlsx")


In [20]:

# Define X and y
X = df.drop(columns=["House CPI", "Months"])   
X = sm.add_constant(X)       
y = df["House CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())




Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.025
Model:                            OLS   Adj. R-squared:                  0.021
Method:                 Least Squares   F-statistic:                     7.573
Date:                Fri, 10 Oct 2025   Prob (F-statistic):            0.00628
Time:                        15:35:19   Log-Likelihood:                -707.61
No. Observations:                 302   AIC:                             1419.
Df Residuals:                     300   BIC:                             1427.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          1.6494      0.1

In [21]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.755
Method:                 Least Squares   F-statistic:                     463.4
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           3.43e-92
Time:                        15:35:19   Log-Likelihood:                -496.79
No. Observations:                 301   AIC:                             999.6
Df Residuals:                     298   BIC:                             1011.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
lag_y          0.8729      0.029     29.944      0.0

In [22]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'EUR/USD' (p-value = 0.8631)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.756
Method:                 Least Squares   F-statistic:                     929.8
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           9.08e-94
Time:                        15:35:19   Log-Likelihood:                -496.80
No. Observations:                 301   AIC:                             997.6
Df Residuals:                     299   BIC:                             1005.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------

In [23]:

# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.027
Model:                            OLS   Adj. R-squared:                  0.024
Method:                 Least Squares   F-statistic:                     8.356
Date:                Fri, 10 Oct 2025   Prob (F-statistic):            0.00413
Time:                        15:35:19   Log-Likelihood:                -705.37
No. Observations:                 301   AIC:                             1415.
Df Residuals:                     299   BIC:                             1422.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
lag_const       1.6515      0.147     11.242      

In [24]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())




Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.027
Model:                            OLS   Adj. R-squared:                  0.024
Method:                 Least Squares   F-statistic:                     8.356
Date:                Fri, 10 Oct 2025   Prob (F-statistic):            0.00413
Time:                        15:35:19   Log-Likelihood:                -705.37
No. Observations:                 301   AIC:                             1415.
Df Residuals:                     299   BIC:                             1422.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
lag_const       1.6515      

In [25]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.755
Method:                 Least Squares   F-statistic:                     463.4
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           3.48e-92
Time:                        15:35:19   Log-Likelihood:                -496.80
No. Observations:                 301   AIC:                             999.6
Df Residuals:                     298   BIC:                             1011.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                  coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------
lag_y           0.8736      0.029     29.890      

In [26]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_EUR/USD' (p-value = 0.9755)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.757
Model:                            OLS   Adj. R-squared:                  0.756
Method:                 Least Squares   F-statistic:                     929.8
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           9.08e-94
Time:                        15:35:19   Log-Likelihood:                -496.80
No. Observations:                 301   AIC:                             997.6
Df Residuals:                     299   BIC:                             1005.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------

# Guinea

In [27]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataHousingCPI/Guinea_completes"
df = pd.read_excel(x+".xlsx")


In [28]:

# Define X and y
X = df.drop(columns=["House CPI", "Months"])   
X = sm.add_constant(X)       
y = df["House CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.4391)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.3295)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.7043)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.457
Model:                            OLS   Adj. R-squared:                  0.455
Method:                 Least Squares   F-statistic:                     217.4
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           4.20e-36
Time:                        15:35:20   Log-Likelihood:                -1043.0
No. Observations:                 260   AIC:                             2090.
Df Residuals:                     258   BIC:                             2097.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef  

In [29]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.905
Model:                            OLS   Adj. R-squared:                  0.903
Method:                 Least Squares   F-statistic:                     482.6
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          3.94e-127
Time:                        15:35:20   Log-Likelihood:                -813.67
No. Observations:                 259   AIC:                             1639.
Df Residuals:                     253   BIC:                             1661.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
lag_y               

In [30]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Crude oil, average ($/bbl)' (p-value = 0.9988)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.5164)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.8896)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.905
Model:                            OLS   Adj. R-squared:                  0.904
Method:                 Least Squares   F-statistic:                     1218.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          1.54e-131
Time:                        15:35:20   Log-Likelihood:                -813.90
No. Observations:                 259   AIC:                             1634.
Df Residuals:                     256   BIC:                             1644.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef  

In [31]:

# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.414
Model:                            OLS   Adj. R-squared:                  0.405
Method:                 Least Squares   F-statistic:                     44.94
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           1.64e-28
Time:                        15:35:20   Log-Likelihood:                -1049.3
No. Observations:                 259   AIC:                             2109.
Df Residuals:                     254   BIC:                             2126.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_const   

In [32]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.6028)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.3206)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.7651)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.411
Model:                            OLS   Adj. R-squared:                  0.409
Method:                 Least Squares   F-statistic:                     179.6
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           2.08e-31
Time:                        15:35:20   Log-Likelihood:                -1050.0
No. Observations:                 259   AIC:                             2104.
Df Residuals:                     257   BIC:                             2111.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
           

In [33]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.905
Model:                            OLS   Adj. R-squared:                  0.904
Method:                 Least Squares   F-statistic:                     484.7
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          2.39e-127
Time:                        15:35:20   Log-Likelihood:                -813.16
No. Observations:                 259   AIC:                             1638.
Df Residuals:                     253   BIC:                             1660.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_y       

In [34]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.6678)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.5797)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.9316)
Dropping 'lag_const' (p-value = 0.4595)

Final model summary:
                                 OLS Regression Results                                
Dep. Variable:              House CPI   R-squared (uncentered):                   0.915
Model:                            OLS   Adj. R-squared (uncentered):              0.914
Method:                 Least Squares   F-statistic:                              1376.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):                   5.02e-138
Time:                        15:35:20   Log-Likelihood:                         -813.70
No. Observations:                 259   AIC:                                      1631.
Df Residuals:                     257   BIC:                                      1639.
Df Model:                           2                    

# Mali 

In [35]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataHousingCPI/Mali_completes"
df = pd.read_excel(x+".xlsx")


In [36]:

# Define X and y
X = df.drop(columns=["House CPI", "Months"])   
X = sm.add_constant(X)       
y = df["House CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'EUR/USD' (p-value = 0.5041)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.3129)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.1825)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.019
Model:                            OLS   Adj. R-squared:                  0.016
Method:                 Least Squares   F-statistic:                     5.892
Date:                Fri, 10 Oct 2025   Prob (F-statistic):             0.0158
Time:                        15:35:20   Log-Likelihood:                -773.57
No. Observations:                 302   AIC:                             1551.
Df Residuals:                     300   BIC:                             1559.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                               coef    s

In [37]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.668
Model:                            OLS   Adj. R-squared:                  0.662
Method:                 Least Squares   F-statistic:                     118.5
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           2.16e-68
Time:                        15:35:20   Log-Likelihood:                -608.67
No. Observations:                 301   AIC:                             1229.
Df Residuals:                     295   BIC:                             1252.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
lag_y               

In [38]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.9941)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.6597)
Dropping 'EUR/USD' (p-value = 0.5762)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.2455)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.665
Model:                            OLS   Adj. R-squared:                  0.664
Method:                 Least Squares   F-statistic:                     594.7
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           4.58e-73
Time:                        15:35:20   Log-Likelihood:                -609.61
No. Observations:                 301   AIC:                             1223.
Df Residuals:                     299   BIC:                             1231.
Df Model:                           1                                         
Covariance Type:            nonrobust                           

In [39]:

# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.027
Model:                            OLS   Adj. R-squared:                  0.013
Method:                 Least Squares   F-statistic:                     2.024
Date:                Fri, 10 Oct 2025   Prob (F-statistic):             0.0911
Time:                        15:35:20   Log-Likelihood:                -770.33
No. Observations:                 301   AIC:                             1551.
Df Residuals:                     296   BIC:                             1569.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_const   

In [40]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_EUR/USD' (p-value = 0.8582)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.7024)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.026
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     3.983
Date:                Fri, 10 Oct 2025   Prob (F-statistic):             0.0196
Time:                        15:35:20   Log-Likelihood:                -770.43
No. Observations:                 301   AIC:                             1547.
Df Residuals:                     298   BIC:                             1558.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025   

In [41]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.669
Model:                            OLS   Adj. R-squared:                  0.664
Method:                 Least Squares   F-statistic:                     119.5
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           9.49e-69
Time:                        15:35:20   Log-Likelihood:                -607.83
No. Observations:                 301   AIC:                             1228.
Df Residuals:                     295   BIC:                             1250.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_y       

In [42]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.9515)
Dropping 'lag_EUR/USD' (p-value = 0.2909)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.1666)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.4890)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.665
Model:                            OLS   Adj. R-squared:                  0.664
Method:                 Least Squares   F-statistic:                     594.7
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           4.58e-73
Time:                        15:35:20   Log-Likelihood:                -609.61
No. Observations:                 301   AIC:                             1223.
Df Residuals:                     299   BIC:                             1231.
Df Model:                           1                                         
Covariance Type:            nonrobust           

# Niger

In [43]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataHousingCPI/Niger_completes"
df = pd.read_excel(x+".xlsx")


In [44]:

# Define X and y
X = df.drop(columns=["House CPI", "Months"])   
X = sm.add_constant(X)       
y = df["House CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Crude oil, average ($/bbl)' (p-value = 0.4963)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.250
Model:                            OLS   Adj. R-squared:                  0.243
Method:                 Least Squares   F-statistic:                     33.18
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           1.57e-18
Time:                        15:35:20   Log-Likelihood:                -731.71
No. Observations:                 302   AIC:                             1471.
Df Residuals:                     298   BIC:                             1486.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------

In [45]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.810
Model:                            OLS   Adj. R-squared:                  0.807
Method:                 Least Squares   F-statistic:                     251.7
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          3.78e-104
Time:                        15:35:21   Log-Likelihood:                -523.13
No. Observations:                 301   AIC:                             1058.
Df Residuals:                     295   BIC:                             1080.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
lag_y               

In [46]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'EUR/USD' (p-value = 0.9924)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.4882)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.810
Model:                            OLS   Adj. R-squared:                  0.808
Method:                 Least Squares   F-statistic:                     421.5
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          1.14e-106
Time:                        15:35:21   Log-Likelihood:                -523.37
No. Observations:                 301   AIC:                             1055.
Df Residuals:                     297   BIC:                             1070.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]


In [47]:

# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.234
Model:                            OLS   Adj. R-squared:                  0.224
Method:                 Least Squares   F-statistic:                     22.65
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           2.44e-16
Time:                        15:35:21   Log-Likelihood:                -732.96
No. Observations:                 301   AIC:                             1476.
Df Residuals:                     296   BIC:                             1494.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_const   

In [48]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.6060)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.234
Model:                            OLS   Adj. R-squared:                  0.226
Method:                 Least Squares   F-statistic:                     30.19
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           4.60e-17
Time:                        15:35:21   Log-Likelihood:                -733.10
No. Observations:                 301   AIC:                             1474.
Df Residuals:                     297   BIC:                             1489.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                   coef    std err          t      P>|t|      [0.025      0.975]
------------------------------

In [49]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.807
Model:                            OLS   Adj. R-squared:                  0.804
Method:                 Least Squares   F-statistic:                     246.4
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          4.79e-103
Time:                        15:35:21   Log-Likelihood:                -525.72
No. Observations:                 301   AIC:                             1063.
Df Residuals:                     295   BIC:                             1086.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_y       

In [50]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.9429)
Dropping 'lag_EUR/USD' (p-value = 0.8992)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.6742)
Dropping 'lag_const' (p-value = 0.1070)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.1455)

Final model summary:
                                 OLS Regression Results                                
Dep. Variable:              House CPI   R-squared (uncentered):                   0.824
Model:                            OLS   Adj. R-squared (uncentered):              0.823
Method:                 Least Squares   F-statistic:                              1405.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):                   3.27e-115
Time:                        15:35:21   Log-Likelihood:                         -528.21
No. Observations:                 301   AIC:                                      1058.
Df Residuals:                     300   BIC:                                      1062.
Df Model:      

# Senegal

In [51]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataHousingCPI/Senegal_completes"
df = pd.read_excel(x+".xlsx")


In [52]:

# Define X and y
X = df.drop(columns=["House CPI", "Months"])   
X = sm.add_constant(X)       
y = df["House CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.7033)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.051
Model:                            OLS   Adj. R-squared:                  0.042
Method:                 Least Squares   F-statistic:                     5.356
Date:                Fri, 10 Oct 2025   Prob (F-statistic):            0.00132
Time:                        15:35:21   Log-Likelihood:                -590.29
No. Observations:                 302   AIC:                             1189.
Df Residuals:                     298   BIC:                             1203.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------

In [53]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.887
Model:                            OLS   Adj. R-squared:                  0.885
Method:                 Least Squares   F-statistic:                     462.3
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          3.05e-137
Time:                        15:35:21   Log-Likelihood:                -268.73
No. Observations:                 301   AIC:                             549.5
Df Residuals:                     295   BIC:                             571.7
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
lag_y               

In [54]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'Crude oil, average ($/bbl)' (p-value = 0.9398)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.8581)
Dropping 'EUR/USD' (p-value = 0.2747)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.1991)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.886
Model:                            OLS   Adj. R-squared:                  0.885
Method:                 Least Squares   F-statistic:                     2317.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          7.14e-143
Time:                        15:35:21   Log-Likelihood:                -270.19
No. Observations:                 301   AIC:                             544.4
Df Residuals:                     299   BIC:                             551.8
Df Model:                           1                                         
Covariance Type:            nonrobust                           

In [55]:

# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.050
Model:                            OLS   Adj. R-squared:                  0.037
Method:                 Least Squares   F-statistic:                     3.881
Date:                Fri, 10 Oct 2025   Prob (F-statistic):            0.00434
Time:                        15:35:21   Log-Likelihood:                -588.95
No. Observations:                 301   AIC:                             1188.
Df Residuals:                     296   BIC:                             1206.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_const   

In [56]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.6108)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.049
Model:                            OLS   Adj. R-squared:                  0.039
Method:                 Least Squares   F-statistic:                     5.101
Date:                Fri, 10 Oct 2025   Prob (F-statistic):            0.00186
Time:                        15:35:21   Log-Likelihood:                -589.09
No. Observations:                 301   AIC:                             1186.
Df Residuals:                     297   BIC:                             1201.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------

In [57]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.886
Model:                            OLS   Adj. R-squared:                  0.885
Method:                 Least Squares   F-statistic:                     460.7
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          4.89e-137
Time:                        15:35:21   Log-Likelihood:                -269.21
No. Observations:                 301   AIC:                             550.4
Df Residuals:                     295   BIC:                             572.7
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_y       

In [58]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.6924)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.8338)
Dropping 'lag_EUR/USD' (p-value = 0.5155)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.2535)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.886
Model:                            OLS   Adj. R-squared:                  0.885
Method:                 Least Squares   F-statistic:                     2317.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          7.14e-143
Time:                        15:35:21   Log-Likelihood:                -270.19
No. Observations:                 301   AIC:                             544.4
Df Residuals:                     299   BIC:                             551.8
Df Model:                           1                                         
Covariance Type:            nonrobust           

# Togo

In [59]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataHousingCPI/Togo_completes"
df = pd.read_excel(x+".xlsx")


In [60]:

# Define X and y
X = df.drop(columns=["House CPI", "Months"])   
X = sm.add_constant(X)       
y = df["House CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.9238)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.6616)
Dropping 'EUR/USD' (p-value = 0.4294)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.1363)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                      -0.000
Model:                            OLS   Adj. R-squared:                 -0.000
Method:                 Least Squares   F-statistic:                       nan
Date:                Fri, 10 Oct 2025   Prob (F-statistic):                nan
Time:                        15:35:21   Log-Likelihood:                -695.30
No. Observations:                 302   AIC:                             1393.
Df Residuals:                     301   BIC:                             1396.
Df Model:                           0                                         
Covariance Type:            nonrobust                           

In [61]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.821
Model:                            OLS   Adj. R-squared:                  0.818
Method:                 Least Squares   F-statistic:                     269.8
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          9.03e-108
Time:                        15:35:21   Log-Likelihood:                -434.88
No. Observations:                 301   AIC:                             881.8
Df Residuals:                     295   BIC:                             904.0
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
lag_y               

In [62]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())


Dropping 'EUR/USD' (p-value = 0.5041)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.3465)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.3699)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.3604)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.819
Model:                            OLS   Adj. R-squared:                  0.818
Method:                 Least Squares   F-statistic:                     1351.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          6.53e-113
Time:                        15:35:21   Log-Likelihood:                -436.39
No. Observations:                 301   AIC:                             876.8
Df Residuals:                     299   BIC:                             884.2
Df Model:                           1                                         
Covariance Type:            nonrobust                           

In [63]:

# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.014
Model:                            OLS   Adj. R-squared:                  0.000
Method:                 Least Squares   F-statistic:                     1.015
Date:                Fri, 10 Oct 2025   Prob (F-statistic):              0.400
Time:                        15:35:22   Log-Likelihood:                -691.39
No. Observations:                 301   AIC:                             1393.
Df Residuals:                     296   BIC:                             1411.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_const   

In [64]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.9369)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.7888)
Dropping 'lag_EUR/USD' (p-value = 0.4377)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.011
Model:                            OLS   Adj. R-squared:                  0.008
Method:                 Least Squares   F-statistic:                     3.409
Date:                Fri, 10 Oct 2025   Prob (F-statistic):             0.0658
Time:                        15:35:22   Log-Likelihood:                -691.73
No. Observations:                 301   AIC:                             1387.
Df Residuals:                     299   BIC:                             1395.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                            

In [65]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.820
Model:                            OLS   Adj. R-squared:                  0.817
Method:                 Least Squares   F-statistic:                     268.1
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          1.99e-107
Time:                        15:35:22   Log-Likelihood:                -435.69
No. Observations:                 301   AIC:                             883.4
Df Residuals:                     295   BIC:                             905.6
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_y       

In [66]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.8801)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.7904)
Dropping 'lag_EUR/USD' (p-value = 0.4846)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.3688)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.819
Model:                            OLS   Adj. R-squared:                  0.818
Method:                 Least Squares   F-statistic:                     1351.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          6.53e-113
Time:                        15:35:22   Log-Likelihood:                -436.39
No. Observations:                 301   AIC:                             876.8
Df Residuals:                     299   BIC:                             884.2
Df Model:                           1                                         
Covariance Type:            nonrobust           

# UEMOA

In [67]:
# Upload the excel file and save 
x ="/Users/harold/DataAnalyctisandScience/Cointegration_Johansen_test/dataHousingCPI/UEMOA_completes"
df = pd.read_excel(x+".xlsx")


In [68]:

# Define X and y
X = df.drop(columns=["House CPI", "Months"])   
X = sm.add_constant(X)       
y = df["House CPI"]
ols_model, X_selected = backward_elimination(y, X, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'EUR/USD' (p-value = 0.6736)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.3161)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.190
Model:                            OLS   Adj. R-squared:                  0.185
Method:                 Least Squares   F-statistic:                     35.09
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           2.05e-14
Time:                        15:35:22   Log-Likelihood:                -516.10
No. Observations:                 302   AIC:                             1038.
Df Residuals:                     299   BIC:                             1049.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
--

In [69]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")

# 2. Combine everything
data = pd.concat([y, y_lagged, X], axis=1).dropna()


# 3. Current y
y_current = data[y.name]

# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X.columns]], axis=1))

# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.835
Model:                            OLS   Adj. R-squared:                  0.832
Method:                 Least Squares   F-statistic:                     299.2
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          3.09e-113
Time:                        15:35:22   Log-Likelihood:                -275.16
No. Observations:                 301   AIC:                             562.3
Df Residuals:                     295   BIC:                             584.6
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                 coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------------
lag_y               

In [70]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

# 1. Create lag of X
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")
X_lagged=X_lagged.dropna()
model_2 = sm.OLS(y_current, X_lagged).fit()
print(model_2.summary())



Dropping 'EUR/USD' (p-value = 0.9389)
Dropping 'Crude oil, WTI ($/bbl)' (p-value = 0.7871)
Dropping 'Crude oil, average ($/bbl)' (p-value = 0.6744)
Dropping 'Crude oil, Brent ($/bbl)' (p-value = 0.1128)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.834
Model:                            OLS   Adj. R-squared:                  0.833
Method:                 Least Squares   F-statistic:                     1499.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          1.63e-118
Time:                        15:35:22   Log-Likelihood:                -276.56
No. Observations:                 301   AIC:                             557.1
Df Residuals:                     299   BIC:                             564.5
Df Model:                           1                                         
Covariance Type:            nonrobust                           

In [71]:

ols_model, X_selected = backward_elimination(y_current, X_lagged, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())



Dropping 'lag_EUR/USD' (p-value = 0.4518)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.3143)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.194
Model:                            OLS   Adj. R-squared:                  0.189
Method:                 Least Squares   F-statistic:                     35.98
Date:                Fri, 10 Oct 2025   Prob (F-statistic):           1.01e-14
Time:                        15:35:22   Log-Likelihood:                -514.03
No. Observations:                 301   AIC:                             1034.
Df Residuals:                     298   BIC:                             1045.
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025   

In [72]:

# 1. Create lag of y and X
y_lagged = y.shift(1).rename("lag_y")
X_lagged = X.shift(1).rename(columns=lambda x: f"lag_{x}")


# 2. Combine everything
data = pd.concat([y, y_lagged, X_lagged], axis=1).dropna()


# 3. Current y
y_current = data[y.name]   # dependent variable


# 4. Explanatory vars = lag of y + your exogenous X’s
X_reg = sm.add_constant(pd.concat([data["lag_y"], data[X_lagged.columns]], axis=1))


# 5. Estimate AR(1) with exogenous regressors
ols_model = sm.OLS(y_current, X_reg).fit()
print(ols_model.summary())



                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.835
Model:                            OLS   Adj. R-squared:                  0.833
Method:                 Least Squares   F-statistic:                     299.4
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          2.78e-113
Time:                        15:35:22   Log-Likelihood:                -275.05
No. Observations:                 301   AIC:                             562.1
Df Residuals:                     295   BIC:                             584.3
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                                     coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------------
lag_y       

In [73]:

ols_model, X_selected = backward_elimination(y_current, X_reg, significance_level=0.10)

print("\nFinal model summary:")
print(ols_model.summary())

Dropping 'lag_EUR/USD' (p-value = 0.9402)
Dropping 'lag_Crude oil, WTI ($/bbl)' (p-value = 0.8901)
Dropping 'lag_Crude oil, average ($/bbl)' (p-value = 0.5100)
Dropping 'lag_Crude oil, Brent ($/bbl)' (p-value = 0.1118)

Final model summary:
                            OLS Regression Results                            
Dep. Variable:              House CPI   R-squared:                       0.834
Model:                            OLS   Adj. R-squared:                  0.833
Method:                 Least Squares   F-statistic:                     1499.
Date:                Fri, 10 Oct 2025   Prob (F-statistic):          1.63e-118
Time:                        15:35:22   Log-Likelihood:                -276.56
No. Observations:                 301   AIC:                             557.1
Df Residuals:                     299   BIC:                             564.5
Df Model:                           1                                         
Covariance Type:            nonrobust           